# Laboratório — Naive Bayes do cálculo manual ao texto

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/08-naive-bayes-probabilidade-condicional-laboratorio.ipynb)

Neste laboratório você vai:

1. reconstruir a regra do Multinomial Naive Bayes a partir de contagens;
2. verificar suavização e cálculo em log-espaço;
3. selecionar representação e `alpha` sem tocar no teste;
4. auditar termos e erros de um classificador de mensagens;
5. mostrar como features duplicadas podem produzir confiança excessiva;
6. conferir a likelihood do Gaussian Naive Bayes.

O conjunto de teste fica isolado até a configuração final.

## Ambiente e reprodutibilidade

Dependências mínimas: Python 3.10, NumPy 1.24, pandas 1.5, Matplotlib 3.7, SciPy 1.10 e scikit-learn 1.3.

Os documentos são sintéticos, gerados localmente com a semente `20260908`. Não há downloads, credenciais ou dependência de serviços externos.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from scipy.special import logsumexp
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.pipeline import Pipeline

SEED = 20260908
np.set_printoptions(precision=6, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Cálculo manual: prior, likelihood e posterior

O corpus mínimo possui três mensagens por classe. Ajustaremos o vocabulário somente nesse corpus, somaremos as ocorrências por classe e aplicaremos suavização de Laplace (`alpha=1`).

Para a classe $c$ e o termo $j$:

\[
\widehat\theta_{cj}=\frac{N_{cj}+\alpha}{N_c+\alpha V},
\]

onde $V$ é o tamanho do vocabulário, $N_{cj}$ é a contagem do termo na classe e $N_c$ é o total de tokens da classe.

In [ ]:
toy_texts = np.array([
    "gratis premio urgente", "oferta gratis agora", "premio oferta urgente",
    "reuniao projeto amanha", "relatorio projeto equipe", "agenda reuniao equipe",
])
toy_y = np.array([1, 1, 1, 0, 0, 0])
toy_vec = CountVectorizer()
X_toy = toy_vec.fit_transform(toy_texts)
vocab = toy_vec.get_feature_names_out()
alpha = 1.0

class_counts = np.bincount(toy_y)
token_counts = np.vstack([np.asarray(X_toy[toy_y == c].sum(axis=0)).ravel() for c in [0, 1]])
theta = (token_counts + alpha) / (token_counts.sum(axis=1, keepdims=True) + alpha * len(vocab))
priors = class_counts / class_counts.sum()

table = pd.DataFrame(token_counts.T, index=vocab, columns=["normal", "spam"])
print(table.to_string())
print("\nPriors:", priors)
assert np.allclose(theta.sum(axis=1), 1.0)
assert priors.tolist() == [0.5, 0.5]

In [ ]:
query = "gratis urgente"
xq = toy_vec.transform([query]).toarray()[0]
log_scores = np.log(priors) + xq @ np.log(theta).T
manual_log_posterior = log_scores - logsumexp(log_scores)

toy_model = MultinomialNB(alpha=alpha).fit(X_toy, toy_y)
sklearn_log_posterior = toy_model.predict_log_proba(toy_vec.transform([query]))[0]

print("Log-scores não normalizados:", log_scores)
print("Posterior manual:", np.exp(manual_log_posterior))
print("Posterior scikit-learn:", np.exp(sklearn_log_posterior))
print("Predição:", int(toy_model.predict(toy_vec.transform([query]))[0]))
assert np.allclose(manual_log_posterior, sklearn_log_posterior, atol=1e-12)

## 2. Probabilidade zero e underflow

Sem suavização, um termo nunca observado em uma classe atribui likelihood zero àquela classe. Mesmo com probabilidades não nulas, multiplicar centenas delas pode ultrapassar a precisão de ponto flutuante. Somar log-probabilidades resolve o segundo problema.

In [ ]:
gratis_idx = int(np.where(vocab == "gratis")[0][0])
unsmoothed_normal = token_counts[0, gratis_idx] / token_counts[0].sum()
smoothed_normal = theta[0, gratis_idx]

direct_product = np.prod(np.full(500, 0.01))
stable_log_sum = np.log(0.01) * 500

print("P(gratis | normal) sem suavização:", unsmoothed_normal)
print("P(gratis | normal) com alpha=1:", f"{smoothed_normal:.9f}")
print("Produto direto de 500 probabilidades 0,01:", direct_product)
print("Log do mesmo produto:", f"{stable_log_sum:.6f}")
assert unsmoothed_normal == 0.0
assert smoothed_normal > 0.0
assert direct_product == 0.0 and np.isfinite(stable_log_sum)

## 3. Corpus sintético e protocolo

Cada linha é uma mensagem independente. Há termos compartilhados, termos mais prováveis em cada classe e 15% de chance de inserir um termo da classe oposta; isso cria sobreposição e erros realistas. Os rótulos são balanceados.

Separamos 25% como teste antes de explorar representação e `alpha`.

In [ ]:
rng = np.random.default_rng(SEED)
spam_terms = np.array(["oferta", "premio", "gratis", "urgente", "desconto", "clique", "ganhe", "limitado"])
normal_terms = np.array(["reuniao", "projeto", "relatorio", "equipe", "prazo", "agenda", "revisao", "tecnico"])
common_terms = np.array(["hoje", "mensagem", "atualizacao", "confirmar", "informacao", "favor"])

def make_document(label, generator):
    length = int(generator.poisson(7) + 4)
    own, opposite = (spam_terms, normal_terms) if label == 1 else (normal_terms, spam_terms)
    tokens = []
    for _ in range(length):
        draw = generator.random()
        if draw < 0.58:
            tokens.append(generator.choice(own))
        elif draw < 0.85:
            tokens.append(generator.choice(common_terms))
        else:
            tokens.append(generator.choice(opposite))
    return " ".join(tokens)

y = np.tile([0, 1], 600)
texts = np.array([make_document(int(label), rng) for label in y])
order = rng.permutation(len(y))
texts, y = texts[order], y[order]

idx_dev, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED
)
text_dev, text_test = texts[idx_dev], texts[idx_test]
y_dev, y_test = y[idx_dev], y[idx_test]

print("Desenvolvimento:", len(idx_dev), "| Teste reservado:", len(idx_test))
print("Classes em desenvolvimento:", np.bincount(y_dev).tolist())
print("Exemplo:", text_dev[0])
assert set(idx_dev).isdisjoint(idx_test)
assert np.bincount(y_dev).tolist() == [450, 450]
assert np.bincount(y_test).tolist() == [150, 150]

## 4. Seleção sem consultar o teste

Comparamos contagens, presença binária e TF-IDF. Cada vetorizador fica dentro do `Pipeline`, portanto aprende vocabulário e estatísticas somente no treino de cada fold. A métrica primária é macro-F1 e os cinco folds estratificados são idênticos em todas as alternativas.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
representations = [
    CountVectorizer(ngram_range=(1, 1)),
    CountVectorizer(binary=True, ngram_range=(1, 1)),
    TfidfVectorizer(ngram_range=(1, 1)),
]

pipe = Pipeline([
    ("repr", CountVectorizer()),
    ("nb", MultinomialNB()),
])
search = GridSearchCV(
    pipe,
    param_grid={"repr": representations, "nb__alpha": [0.1, 0.5, 1.0, 2.0, 5.0]},
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
search.fit(text_dev, y_dev)

rows = pd.DataFrame(search.cv_results_)
rows["representacao"] = rows["param_repr"].map(
    lambda x: "tf-idf" if isinstance(x, TfidfVectorizer) else ("binaria" if x.binary else "contagem")
)
summary = rows[["representacao", "param_nb__alpha", "mean_train_score", "mean_test_score", "std_test_score"]]
summary = summary.sort_values("mean_test_score", ascending=False)
print(summary.head(10).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\nMelhores parâmetros:", search.best_params_)
print("Macro-F1 CV:", f"{search.best_score_:.6f}")
assert search.best_score_ > 0.90

## 5. Avaliação final, uma única vez

Agora reajustamos a configuração escolhida em todo o desenvolvimento e abrimos o teste. O baseline sempre prevê a classe mais frequente. Macro-F1 e acurácia balanceada medem decisão; log-loss e Brier também respondem à qualidade das probabilidades.

In [ ]:
final_model = search.best_estimator_
final_model.fit(text_dev, y_dev)
test_pred = final_model.predict(text_test)
test_proba = final_model.predict_proba(text_test)[:, 1]

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_dev), 1)), y_dev)
dummy_pred = dummy.predict(np.zeros((len(y_test), 1)))

metrics = {
    "macro_f1": f1_score(y_test, test_pred, average="macro"),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "log_loss": log_loss(y_test, np.c_[1 - test_proba, test_proba]),
    "brier": brier_score_loss(y_test, test_proba),
    "dummy_macro_f1": f1_score(y_test, dummy_pred, average="macro"),
}
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")
assert metrics["macro_f1"] > 0.90
assert metrics["macro_f1"] > metrics["dummy_macro_f1"] + 0.40

## 6. O que o modelo aprendeu?

No Multinomial NB, a diferença entre log-likelihoods por classe mostra quais termos deslocam o score. Isso é associação preditiva, não causalidade: um termo pode funcionar por convenções do corpus e falhar após mudança de domínio.

In [ ]:
vectorizer = final_model.named_steps["repr"]
nb_model = final_model.named_steps["nb"]
feature_names = vectorizer.get_feature_names_out()
log_odds = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]

top_spam = feature_names[np.argsort(log_odds)[-8:][::-1]]
top_normal = feature_names[np.argsort(log_odds)[:8]]
print("Termos associados a spam:", top_spam.tolist())
print("Termos associados a normal:", top_normal.tolist())
assert set(top_spam).intersection(spam_terms)
assert set(top_normal).intersection(normal_terms)

In [ ]:
wrong = np.flatnonzero(test_pred != y_test)
confidence = np.maximum(test_proba, 1 - test_proba)
ordered_wrong = wrong[np.argsort(confidence[wrong])[::-1]]

print("Erros no teste:", len(wrong))
for i in ordered_wrong[:5]:
    print({
        "verdade": int(y_test[i]),
        "predicao": int(test_pred[i]),
        "confianca": round(float(confidence[i]), 4),
        "texto": text_test[i],
    })
assert len(wrong) > 0

## 7. Independência violada e excesso de confiança

Duplicaremos exatamente todas as colunas da representação. Não acrescentamos informação: cada nova feature é cópia perfeita de outra. Mesmo assim, o Naive Bayes conta a evidência novamente, porque a fatoração a trata como se fosse condicionalmente independente.

Com classes balanceadas, a fronteira permanece praticamente a mesma, mas os scores ficam mais extremos.

In [ ]:
X_dev_base = vectorizer.transform(text_dev)
X_test_base = vectorizer.transform(text_test)
X_dev_dup = sparse.hstack([X_dev_base, X_dev_base], format="csr")
X_test_dup = sparse.hstack([X_test_base, X_test_base], format="csr")

alpha_best = float(search.best_params_["nb__alpha"])
base_nb = MultinomialNB(alpha=alpha_best).fit(X_dev_base, y_dev)
dup_nb = MultinomialNB(alpha=alpha_best).fit(X_dev_dup, y_dev)
base_pred = base_nb.predict(X_test_base)
dup_pred = dup_nb.predict(X_test_dup)
base_p = base_nb.predict_proba(X_test_base)[:, 1]
dup_p = dup_nb.predict_proba(X_test_dup)[:, 1]

base_conf = np.maximum(base_p, 1 - base_p).mean()
dup_conf = np.maximum(dup_p, 1 - dup_p).mean()
base_ll = log_loss(y_test, np.c_[1 - base_p, base_p])
dup_ll = log_loss(y_test, np.c_[1 - dup_p, dup_p])

print("Predições diferentes:", int(np.sum(base_pred != dup_pred)))
print("Confiança média — base / duplicada:", f"{base_conf:.6f}", f"{dup_conf:.6f}")
print("Log-loss — base / duplicada:", f"{base_ll:.6f}", f"{dup_ll:.6f}")
assert np.array_equal(base_pred, dup_pred)
assert dup_conf > base_conf

**Figura 1 — descrição acessível:** diagrama de confiabilidade com duas linhas. A diagonal representa calibração ideal; os pontos do modelo original e do modelo com features duplicadas comparam probabilidade média e frequência observada em grupos de tamanho semelhante. O gráfico é diagnóstico, não uma etapa de seleção.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for probs, label, marker in [(base_p, "features originais", "o"), (dup_p, "features duplicadas", "s")]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=6, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker=marker, label=label)
ax.plot([0, 1], [0, 1], "--", color="gray", label="ideal")
ax.set(xlabel="probabilidade prevista média", ylabel="fração positiva observada", title="Evidência duplicada aumenta a confiança")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 8. Gaussian Naive Bayes

Para features contínuas, o Gaussian NB estima média e variância de cada feature em cada classe. A célula abaixo calcula a log-likelihood de uma consulta e confere a posterior com a implementação oficial.

In [ ]:
rng_g = np.random.default_rng(SEED + 1)
X0 = rng_g.normal(loc=[-1.0, 0.0], scale=[0.8, 1.2], size=(250, 2))
X1 = rng_g.normal(loc=[1.0, 1.5], scale=[1.0, 0.7], size=(250, 2))
Xg = np.vstack([X0, X1])
yg = np.r_[np.zeros(250, dtype=int), np.ones(250, dtype=int)]
gnb = GaussianNB().fit(Xg, yg)
q = np.array([[0.4, 1.0]])

means = gnb.theta_
variances = gnb.var_
manual_joint = np.log(gnb.class_prior_) - 0.5 * np.sum(
    np.log(2 * np.pi * variances) + (q - means) ** 2 / variances, axis=1
)
manual_log_post = manual_joint - logsumexp(manual_joint)
library_log_post = gnb.predict_log_proba(q)[0]

print("Médias por classe:\n", means)
print("Variâncias por classe:\n", variances)
print("Posterior manual:", np.exp(manual_log_post))
print("Posterior scikit-learn:", np.exp(library_log_post))
assert np.allclose(manual_log_post, library_log_post, atol=1e-12)

## 9. Conclusões verificadas

- A posterior manual do Multinomial NB coincidiu com o scikit-learn.
- Laplace impediu probabilidade zero; log-espaço impediu underflow.
- Vetorização e `alpha` foram escolhidos por validação cruzada somente em desenvolvimento.
- O teste foi aberto uma vez, após a seleção.
- Termos associados às classes puderam ser auditados.
- Duplicar features manteve as classes previstas, mas elevou a confiança sem acrescentar informação.
- A posterior manual do Gaussian NB também coincidiu com a biblioteca.

### Desafios

1. Implemente Bernoulli NB e inclua explicitamente a ausência dos termos na likelihood.
2. Altere a prevalência de spam e compare prior aprendido com `class_prior` fixo.
3. Aumente a sobreposição entre vocabulários e registre macro-F1, log-loss e os erros mais confiantes.

Não use o teste para escolher a melhor alteração.